In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Najafgarh, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,101.57,128.01,10.08,39.41,29.08,48.45,10.28,1.23,29.88,0.77,0.82,81.95,0.43,262.21,995.79,13.82,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,115.68,150.41,10.28,38.84,29.03,70.64,10.53,1.48,26.92,0.92,1.40,82.96,0.47,255.06,995.02,13.79,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,151.00,183.63,15.33,44.40,36.08,77.06,11.20,2.04,25.83,0.94,2.71,83.16,0.40,215.81,994.82,14.38,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,232.84,268.08,14.25,71.12,49.73,90.08,11.27,3.08,53.70,0.81,2.49,80.53,0.55,198.06,992.70,15.65,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,151.64,194.24,8.16,46.85,31.48,74.63,10.38,1.74,31.79,0.74,1.40,78.53,0.71,195.48,992.51,15.57,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,230.67,313.83,13.33,106.79,67.59,38.13,13.89,1.39,11.87,2.96,6.76,65.94,0.55,252.57,993.10,18.67,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,256.71,342.12,15.80,93.84,62.71,37.20,11.17,1.58,12.21,1.85,8.89,72.55,0.51,228.50,994.82,17.43,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,230.29,307.58,14.02,109.87,69.84,35.63,15.12,1.37,10.04,2.05,8.90,71.88,0.43,229.92,994.61,17.24,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,217.62,297.83,16.10,115.03,74.28,36.29,18.83,1.44,10.24,2.35,10.63,65.36,0.49,238.80,995.94,18.60,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date    PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  101.570  128.01  10.08  39.41  29.08   
1  02-01-2025 00:00  03-01-2025 00:00  115.680  150.41  10.28  38.84  29.03   
2  03-01-2025 00:00  04-01-2025 00:00  151.000  183.63  15.33  44.40  36.08   
3  04-01-2025 00:00  05-01-2025 00:00   57.065  268.08  14.25  71.12  49.73   
4  05-01-2025 00:00  06-01-2025 00:00  151.640  194.24   8.16  46.85  31.48   

     NH3    SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  48.45  10.28  1.23  29.88     0.77     0.82  81.95  0.43  262.21  995.79   
1  32.49  10.53  1.48  26.92     0.92     1.40  82.96  0.47  255.06  995.02   
2  32.49  11.20  0.76  25.83     0.94     0.29  83.16  0.40  215.81  994.82   
3  32.49  11.27  0.76  53.70     0.81     0.29  80.53  0.55  198.06  992.70   
4  32.49  10.38  1.74  31.79     0.74     1.40  78.53  0.71  195.48  992.51   

      AT   RF  TOT-RF  
0  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,1.380549,-0.031911,0.248574,-0.370636,-0.247432,1.398615,-0.091864,1.224175,-0.631409,1.042045,1.507981,0.895389,-1.154080,1.230780,1.717171,-2.286167,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,1.820757,0.304309,0.281510,-0.392884,-0.250256,0.151144,-0.035579,1.944915,-0.722984,1.404732,3.372614,0.974369,-0.941606,1.022269,1.599899,-2.291857,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,2.922680,0.802936,1.113145,-0.175869,0.147945,0.151144,0.115264,-0.130814,-0.756706,1.453091,-0.195907,0.990008,-1.313435,-0.122356,1.569439,-2.179949,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.007931,2.070517,0.935291,0.867053,0.918932,0.151144,0.131024,-0.130814,0.105520,1.138762,-0.195907,0.784349,-0.516659,-0.639989,1.246560,-1.939063,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.942647,0.962191,-0.067611,-0.080241,-0.111874,0.151144,-0.069350,2.694483,-0.572319,0.969508,3.372614,0.627953,0.333235,-0.715228,1.217623,-1.954237,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.007931,2.757217,0.783785,2.259306,1.927709,0.591980,0.720889,1.685449,-1.188592,-0.336166,-0.195907,-0.356554,-0.516659,0.949654,1.307481,-1.366246,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.007931,-0.168575,1.190545,1.753848,1.652075,0.519289,0.108510,2.233210,-1.178073,3.653392,-0.195907,0.160332,-0.729133,0.247715,1.569439,-1.601442,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.007931,2.663406,0.897414,2.379523,2.054795,0.396574,0.997810,1.627789,-1.245208,4.136975,-0.195907,0.107940,-1.154080,0.289125,1.537456,-1.637480,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.007931,2.517060,1.239949,2.580926,2.305577,0.448161,1.833077,1.829596,-1.239020,-0.336166,-0.195907,-0.401909,-0.835370,0.548088,1.740016,-1.379523,0.0,0.0


In [10]:
df.to_excel('Najafgarh2025.xlsx', index=False)